# Prediksi Sentimen dengan SMOTE (Phase 4 + SMOTE)

Menggunakan **Logistic Regression** (LR) dan **Naive Bayes** (NB) dilatih pada
gabungan `comments_sentiment` + `phase2_auto_labeled` dengan **SMOTE** pada kelas positif.
Memprediksi 3 komentar yang sudah diedit agar lebih mudah ditebak:

1. *Positif*: komentar berisi kata "bagus", "setuju", "bermanfaat"
2. *Negatif*: komentar berisi kata "tidak setuju", "buruk", "menolak"
3. *Netral*: komentar berisi kata "informasi", "mempelajari"


## 1. Import & Konfigurasi


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer, IDF, NGram, RegexTokenizer,
    StringIndexer, VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType


In [2]:
TEXT_COL = "text_final"
LABEL_COL = "sentiment"
VALID_LABELS = ("positif", "netral", "negatif")
SEED = 42

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(dotenv_path=PROJECT_ROOT / ".env", override=True)
MONGO_URI = os.getenv("MONGO_URI", "").strip()
MONGO_DB = os.getenv("MONGO_DB", "analisis_sentimen").strip()
MONGO_LABELED_COLLECTION = "comments_sentiment"
MONGO_AUTO_LABELED_COLLECTION = "phase2_auto_labeled"

print("Konfigurasi siap.")


Konfigurasi siap.


In [3]:
def create_spark_session(app_name: str = "prediksi-komentar") -> SparkSession:
    java_home = os.environ.get("JAVA_HOME") or r"C:\\Program Files\\Java\\jdk-22"
    os.environ["JAVA_HOME"] = java_home
    os.environ["PYSPARK_PYTHON"] = os.environ.get("PYSPARK_PYTHON") or "python"
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.driver.memory", "8g") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.default.parallelism", "4") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .getOrCreate()
    spark._jsc.hadoopConfiguration().set(
        "fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem"
    )
    return spark


def _normalize_value(v):
    if v is None:
        return ''
    if isinstance(v, (int, float)):
        return str(v)
    return str(v)


def build_pipeline(model_name: str, **kwargs) -> Pipeline:
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol="tokens",
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol="label_index", handleInvalid="keep"
    )
    ngram = NGram(n=2, inputCol="tokens", outputCol="bigrams")
    name = model_name.lower().strip()
    vocab_uni = kwargs.get("vocab_uni", 8000)
    vocab_bi = kwargs.get("vocab_bi", 6000)
    min_df = kwargs.get("min_df", 3.0)
    cv_uni = CountVectorizer(
        inputCol="tokens", outputCol="uni_feat",
        vocabSize=vocab_uni, minDF=min_df, minTF=1,
    )
    cv_bi = CountVectorizer(
        inputCol="bigrams", outputCol="bi_feat",
        vocabSize=vocab_bi, minDF=min_df, minTF=1,
    )

    if name in ("logistic_regression", "lr", "logistic regression"):
        assembler = VectorAssembler(inputCols=["uni_feat", "bi_feat"], outputCol="raw_feat")
        idf = IDF(inputCol="raw_feat", outputCol="features", minDocFreq=2)
        clf = LogisticRegression(
            featuresCol="features", labelCol="label_index", predictionCol="pred_index",
            maxIter=kwargs.get("max_iter", 300), regParam=kwargs.get("reg_param", 0.05),
            elasticNetParam=kwargs.get("elastic_net", 0.15), family="multinomial", tol=1e-4,
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]

    elif name in ("naive_bayes", "nb", "naive bayes"):
        use_bigrams = kwargs.get("use_bigrams", False)
        clf = NaiveBayes(
            featuresCol="features", labelCol="label_index", predictionCol="pred_index",
            modelType="multinomial", smoothing=kwargs.get("smoothing", 0.5),
        )
        if use_bigrams:
            assembler = VectorAssembler(inputCols=["uni_feat", "bi_feat"], outputCol="features")
            stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, label_indexer, clf]
        else:
            assembler = VectorAssembler(inputCols=["uni_feat"], outputCol="features")
            stages = [tokenizer, cv_uni, assembler, label_indexer, clf]

    else:
        raise ValueError(f"Model tidak dikenal: {model_name}")
    return Pipeline(stages=stages)


spark = create_spark_session()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} siap")


Spark 4.1.2 siap


## 2. Load & Gabung Data dari MongoDB (Phase 4)


In [ ]:
import json

def _load_collection(collection_name):
    docs = []
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        cursor = client[MONGO_DB][collection_name].find(
            {TEXT_COL: {"$exists": True, "$nin": ["", None]},
             LABEL_COL: {"$exists": True, "$nin": [None, ""]}},
            {"_id": 0, "comment_id": 1, TEXT_COL: 1, LABEL_COL: 1},
        )
        for doc in cursor:
            docs.append({
                "comment_id": doc.get("comment_id", ""),
                TEXT_COL: _normalize_value(doc[TEXT_COL]),
                LABEL_COL: doc.get(LABEL_COL, ""),
            })
    return docs

dfo_docs = _load_collection(MONGO_LABELED_COLLECTION)
dfa_docs = _load_collection(MONGO_AUTO_LABELED_COLLECTION)
if not dfo_docs:
    raise RuntimeError(f"Tidak ada data di {MONGO_LABELED_COLLECTION}")
print(f"Original ({MONGO_LABELED_COLLECTION}): {len(dfo_docs)}")
print(f"Auto-labeled ({MONGO_AUTO_LABELED_COLLECTION}): {len(dfa_docs) if dfa_docs else 0}")

dfo = spark.createDataFrame(dfo_docs).cache()
if dfa_docs:
    dfa = spark.createDataFrame(dfa_docs).cache()
    common_cols = [c for c in dfo.columns if c in dfa.columns]
    df_all = dfo.select(*common_cols).unionByName(dfa.select(*common_cols)).cache()
else:
    df_all = dfo.cache()

print(f"Total data latih: {df_all.count()} baris")
df_all.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
train_df = df_all


Original (comments_sentiment): 1471
Auto-labeled (phase2_auto_labeled): 11706


## 3. Latih Model Logistic Regression (LR) + SMOTE


In [ ]:
import numpy as np
from pyspark.ml.linalg import Vectors
from imblearn.over_sampling import SMOTE

print(">>> Melatih Logistic Regression + SMOTE...")

# Feature pipeline (Spark)
lr_pipeline_obj = build_pipeline("lr")
lr_stages = lr_pipeline_obj.getStages()
feat_stages = lr_stages[:-1]
feat_model = Pipeline(stages=feat_stages).fit(train_df)
train_feat = feat_model.transform(train_df).cache()

# Distribusi sebelum SMOTE
print(f"Sebelum SMOTE: {train_feat.count()} baris")
train_feat.groupBy("label_index").count().orderBy("label_index").show()
si_model = feat_model.stages[-1]
labels = list(si_model.labels)
print(f"Label mapping: {labels}")

# Konversi ke numpy untuk SMOTE + sklearn
train_pd = train_feat.select("features", "label_index").toPandas()
X = np.array([v.toArray() for v in train_pd["features"]])
y = train_pd["label_index"].values.astype(int)

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_res, y_res = smote.fit_resample(X, y)
print(f"Setelah SMOTE: {len(X_res)} baris")
for i, lbl in enumerate(labels):
    cnt = int((y_res == i).sum())
    print(f"  {lbl} (index {i}): {cnt}")

# Latih sklearn LogisticRegression + GridSearch pada data SMOTE
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.model_selection import GridSearchCV as SkGridSearch

param_grid = {'C': [0.1, 0.5, 1.0, 5.0, 10.0]}
gs = SkGridSearch(
    SkLR(max_iter=500, multi_class='multinomial', solver='saga', tol=1e-4, random_state=SEED),
    param_grid, cv=3, scoring='f1_macro', n_jobs=4,
)
gs.fit(X_res, y_res)
best_lr_sk = gs.best_estimator_
print(f"Best LR (sklearn): C={gs.best_params_['C']:.4f}")
print("Model LR + SMOTE siap.")


## 4. Latih Model Naive Bayes (NB) + SMOTE


In [ ]:
print(">>> Melatih Naive Bayes + SMOTE...")

nb_pipeline = build_pipeline("nb", smoothing=0.5, use_bigrams=False)
nb_stages = nb_pipeline.getStages()
nb_feat_stages = nb_stages[:-1]
nb_feat_model = Pipeline(stages=nb_feat_stages).fit(train_df)
nb_train_feat = nb_feat_model.transform(train_df).cache()

# SMOTE pada feature NB
nb_train_pd = nb_train_feat.select("features", "label_index").toPandas()
X_nb = np.array([v.toArray() for v in nb_train_pd["features"]])
y_nb = nb_train_pd["label_index"].values.astype(int)

smote_nb = SMOTE(random_state=SEED, k_neighbors=5)
X_nb_res, y_nb_res = smote_nb.fit_resample(X_nb, y_nb)
print(f"Setelah SMOTE NB: {len(X_nb_res)} baris")
si_model_nb = nb_feat_model.stages[-1]
nb_labels = list(si_model_nb.labels)
for i, lbl in enumerate(nb_labels):
    cnt = int((y_nb_res == i).sum())
    print(f"  {lbl} (index {i}): {cnt}")

# Latih sklearn MultinomialNB pada data balanced
from sklearn.naive_bayes import MultinomialNB
nb_sk = MultinomialNB(alpha=0.5)
nb_sk.fit(X_nb_res, y_nb_res)
print(f"NB labels: {nb_labels}")
print("Model NB + SMOTE siap.")


## 5. Siapkan Data Uji (3 Komentar)


In [ ]:
test_comments = [
    {
        "comment_id": "komentar_1",
        "text_final": "Video ini sangat membantu dan membuka wawasan saya tentang RUU TNI Penjelasannya bagus jelas dan mudah dipahami Saya setuju dengan apa yang disampaikan Terima kasih untuk informasinya yang bermanfaat",
    },
    {
        "comment_id": "komentar_2",
        "text_final": "Saya sangat tidak setuju dengan RUU TNI ini Kebijakan ini buruk dan akan merusak reformasi Militer tidak boleh masuk ke jabatan sipil karena berbahaya bagi demokrasi Saya menolak RUU ini",
    },
    {
        "comment_id": "komentar_3",
        "text_final": "Saya membaca video ini untuk menambah informasi Penjelasan yang diberikan cukup lengkap Saya ingin mempelajari lebih lanjut tentang topik ini dari berbagai sumber",
    },
]

test_df = spark.createDataFrame(test_comments)
test_df.show(truncate=80)


## 6. Prediksi dengan Kedua Model (SMOTE)


In [ ]:
def predict_sklearn(feat_model, sk_model, test_df, labels):
    dummy = test_df.withColumn(LABEL_COL, F.lit("netral"))
    feats = feat_model.transform(dummy).drop(LABEL_COL)
    test_pd = feats.select("comment_id", "features").toPandas()
    X_test = np.array([v.toArray() for v in test_pd["features"]])
    y_pred = sk_model.predict(X_test)
    y_prob = sk_model.predict_proba(X_test)
    results = []
    for i in range(len(test_pd)):
        row = {"comment_id": test_pd.iloc[i]["comment_id"]}
        row["pred_label"] = labels[y_pred[i]]
        for j, lbl in enumerate(labels):
            row[f"prob_{lbl}"] = float(y_prob[i][j])
        results.append(row)
    return results


lr_result = predict_sklearn(feat_model, best_lr_sk, test_df, labels)
nb_result = predict_sklearn(nb_feat_model, nb_sk, test_df, nb_labels)

print("=== Prediksi Logistic Regression (SMOTE) ===")
for r in lr_result:
    print(f"  {r['comment_id']}: {r['pred_label']}  (pos: {r['prob_positif']:.4f}, net: {r['prob_netral']:.4f}, neg: {r['prob_negatif']:.4f})")

print("\n=== Prediksi Naive Bayes (SMOTE) ===")
for r in nb_result:
    print(f"  {r['comment_id']}: {r['pred_label']}  (pos: {r['prob_positif']:.4f}, net: {r['prob_netral']:.4f}, neg: {r['prob_negatif']:.4f})")


## 7. Tabel Perbandingan (SMOTE)


In [ ]:
lr_dict = {r["comment_id"]: r for r in lr_result}
nb_dict = {r["comment_id"]: r for r in nb_result}

print("=" * 110)
print(f"{'Komentar':<12} {'Model':>7} {'Prediksi':<12} {'Prob Positif':<14} {'Prob Netral':<14} {'Prob Negatif':<14}  Keterangan")
print("=" * 110)

targets = {"komentar_1": "target: positif", "komentar_2": "target: negatif", "komentar_3": "target: netral"}
for cid in ["komentar_1", "komentar_2", "komentar_3"]:
    lr = lr_dict[cid]
    nb = nb_dict[cid]
    print(f"{cid:<12} {'LR':>7} {lr['pred_label']:<12} {lr['prob_positif']:<14.4f} {lr['prob_netral']:<14.4f} {lr['prob_negatif']:<14.4f}  {targets[cid]}")
    print(f"{'':12} {'NB':>7} {nb['pred_label']:<12} {nb['prob_positif']:<14.4f} {nb['prob_netral']:<14.4f} {nb['prob_negatif']:<14.4f}  {'':20}")
    print("-" * 110)

print()
print("Keterangan:")
print("  komentar_1 (target: positif): kata kunci 'bagus, setuju, membantu, bermanfaat, terima kasih'")
print("  komentar_2 (target: negatif): kata kunci 'tidak setuju, buruk, merusak, menolak, berbahaya'")
print("  komentar_3 (target: netral): kata kunci 'informasi, membaca, mempelajari, lengkap'")
print("\nModel dilatih dengan SMOTE (semua kelas seimbang 10401 sampel).")


In [ ]:
spark.stop()
print("Selesai.")
